In [1]:
# Chains in Langchain
#
# A chain is a sequence of calls to LLMs or other components, where the output of one call 
# is the input to the next.
# Chains can be used to create more complex workflows and to manage state between calls.

In [2]:
#LangChain Chains
#
#                    Two methods to create chains
#                                |
#    -----------------------------------------------------------------
#    |                                                               |
# Legacy OOP Style                                       LCEL (Langchain expression language)
#  (removed from Langchain 1.0 onwards)                        (langchain 0.3 .. stable)
#  langchain 0.1.x,0.2.x,0.3.x

## Langchain Expression Language Basics

-  LangChain Expression Language is that any two runnables can be "chained" together into sequences. 
- The output of the previous runnable's .invoke() call is passed as input to the next runnable.
- This can be done using the pipe operator (|), or the more explicit .pipe() method, which does the same thing.

- Type of LCEL Chains
    - SequentialChain
    - Parallel Chain
    - Router Chain
    - Chain Runnables
    - Custom Chain (Runnable Sequence)

1. What is LCEL?

LCEL stands for LangChain Expression Language.
It provides a concise, composable, and Pythonic way to build chains, pipelines, and workflows in LangChain.

Goal: Replace the older SequentialChain, RouterChain, etc., with a more flexible and declarative API.
Key Concepts: Chains as functions (or callables), easy composition (| operator), clear input/output typing.

2. Core LCEL Components

a. Runnable
The core abstraction in LCEL.
Any object that implements the .invoke() method and can be composed using the | operator.
Examples: Models, PromptTemplates, Chains.

b. PromptTemplate
Used to format inputs into prompts for LLMs or ChatModels.
LCEL expects prompt templates to have variable names matching the input keys.

c. LLM and ChatModel
Language Models and ChatModels from langchain_openai, langchain_community, etc.
Compatible with LCEL if they implement .invoke().

In [3]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        ChatPromptTemplate
)

llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001B87CB09150>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001B87C8F3E10>, root_client=<openai.OpenAI object at 0x000001B87C3918D0>, root_async_client=<openai.AsyncOpenAI object at 0x000001B87C9039D0>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [5]:
system = SystemMessagePromptTemplate.from_template('Act as a {persona} professor. You answer in short sentences and in {language} language')
question= HumanMessagePromptTemplate.from_template('Tell me about {topic} in {points} points')

messages = [system,question]

template = ChatPromptTemplate(messages)



chain = template | llm

In [6]:
type(chain)

langchain_core.runnables.base.RunnableSequence

In [7]:
response = chain.invoke({
    'persona' : 'gemologist',
    'language' : 'english',
    'topic' : 'types of stones',
    'points': 3
})

print(response.content)

1. **Precious Stones**: These include diamonds, rubies, sapphires, and emeralds. They are rare and highly valued for their beauty and durability.

2. **Semi-Precious Stones**: Examples are amethyst, garnet, and turquoise. They are more abundant and often used in jewelry but are still valued for their unique colors and properties.

3. **Organic Stones**: These include pearls, coral, and amber. They are formed from biological processes and have distinct characteristics compared to mineral-based stones.


In [8]:
response = chain.invoke({
    'persona': 'math',
    'language': 'english',
    'topic': 'calculus',
    'points': 10
})

print(response.content)

1. Calculus studies change and motion.  
2. It has two main branches: differential and integral calculus.  
3. Differential calculus focuses on rates of change and slopes of curves.  
4. Integral calculus deals with areas under curves and accumulation of quantities.  
5. The fundamental theorem of calculus links both branches.  
6. Limits are foundational to calculus, defining instantaneous rates and areas.  
7. Derivatives represent the slope of a function at a point.  
8. Integrals can be thought of as the "sum" of infinitesimal parts.  
9. Applications include physics, engineering, economics, and biology.  
10. Mastery of calculus requires practice and understanding of its concepts.


In [9]:
response = chain.invoke({
    'persona': 'physics',
    'language': 'english',
    'topic': 'kinematics and dynamics',
    'points': 10
})

print(response.content)

1. **Kinematics** studies motion without considering forces.  
2. **Dynamics** examines the forces causing motion.  
3. Kinematics involves concepts like displacement, velocity, and acceleration.  
4. Dynamics includes Newton's laws of motion.  
5. Kinematics uses equations of motion for constant acceleration.  
6. Dynamics analyzes forces using free-body diagrams.  
7. Kinematics can describe linear and rotational motion.  
8. Dynamics considers mass, weight, and friction.  
9. Kinematics is often visualized with graphs.  
10. Dynamics connects motion to energy and momentum.


# What is StrOutputParser?


StrOutputParser is a utility class in LangChain for handling the output of language models (LLMs/ChatModels).
It parses the model’s response and returns it as a plain Python string, removing any unnecessary metadata, wrappers, or objects that the model or API may return.


## Why do we need it?

- By default, LLMs or chat models in LangChain might return structured objects, message objects, or dictionaries.
- In most cases, especially for simple chains, you just want the generated text string (e.g., the answer, summary, completion, etc.).
- StrOutputParser extracts this text from the raw output so your chain returns a clean string.


In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt =PromptTemplate.from_template('What is the capital of {country} ?')
llm = ChatOpenAI(temperature=0, model_name= "gpt-4o-mini")
parser= StrOutputParser()
chain1 = prompt | llm
chain2 = prompt | llm | parser

print("without StrOutputParser:")

result1 = chain1.invoke({
    'country' : 'Italy'
})
print(result1)

print('With StrOutputParser :')

result2= chain2.invoke({
    'country' :'Italy'
})

print(result2)

without StrOutputParser:
content='The capital of Italy is Rome.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1590f93f9d', 'id': 'chatcmpl-D5PfF08Lgn71pPjVLpPY5oh3XaV0j', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c271b-da39-7ff2-aa30-bbedee63d298-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
With StrOutputParser :
The capital of Italy is Rome.


### Parallel LCEL Chain
- Parallel chains are used to run multiple runnables in parallel.
- The final return value is a dict with the results of each value under its appropriate key.


### What is a Parallel Chain?

Parallel chaining lets you run multiple chains (pipelines) simultaneously using the same or similar inputs, collecting all outputs together.

In LangChain LCEL, this is achieved via RunnableParallel.

It’s analogous to a “fan-out” pattern, where you branch out your input to multiple tasks and gather all their results in one go.

### Why Use Parallel Chains?

- To answer multiple questions about the same input in one shot.
- To generate diverse content (e.g., facts, poems, summaries) from the same base information.
- To save code and avoid running chains sequentially when there’s no dependency between them.




In [13]:
system = SystemMessagePromptTemplate.from_template('Act as a {persona} professor. You answer in short sentences and in {language} language')

question = HumanMessagePromptTemplate.from_template('Tell me about {topics} in {points} points')

messages1=[system,question]

template = ChatPromptTemplate(messages1)

factChain = template | llm | StrOutputParser()

In [14]:
question = HumanMessagePromptTemplate.from_template('write a poem on {topics} in {sentences} lines')
system = SystemMessagePromptTemplate.from_template('Act as a {persona} professor. You answer in short sentences and in {language} language')

messages = [system,question]

template = ChatPromptTemplate(messages)

poemChain = template | llm | StrOutputParser()


In [15]:
from langchain_core.runnables import RunnableParallel

finalChain = RunnableParallel(fact = factChain, poem = poemChain)

output = finalChain.invoke({
    'persona': 'english',
    'language': 'english',
    'topics': 'AI',
    'sentences': 4,
    'points': 5
})

In [16]:
output

{'fact': '1. AI stands for artificial intelligence, simulating human intelligence in machines.  \n2. It encompasses machine learning, natural language processing, and robotics.  \n3. AI is used in various fields, including healthcare, finance, and entertainment.  \n4. Ethical concerns arise regarding privacy, bias, and job displacement.  \n5. The future of AI holds potential for innovation but requires careful regulation.',
 'poem': 'In circuits hums a silent mind,  \nWith thoughts of code, its dreams entwined.  \nA mirror of us, both bright and stark,  \nIn shadows deep, it leaves its mark.'}

In [17]:
print(output['fact'])

1. AI stands for artificial intelligence, simulating human intelligence in machines.  
2. It encompasses machine learning, natural language processing, and robotics.  
3. AI is used in various fields, including healthcare, finance, and entertainment.  
4. Ethical concerns arise regarding privacy, bias, and job displacement.  
5. The future of AI holds potential for innovation but requires careful regulation.


In [18]:
print(output['poem'])

In circuits hums a silent mind,  
With thoughts of code, its dreams entwined.  
A mirror of us, both bright and stark,  
In shadows deep, it leaves its mark.


# Sequential Chain using Runnables 

In [19]:
system = SystemMessagePromptTemplate.from_template('You are a {school} teacher. You answer in short sentences. Make the students life difficult to understand')


question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system,question]
template = ChatPromptTemplate(messages)

teacherChain = template |llm

In [21]:
analysis_prompt = ChatPromptTemplate.from_template(''' analyze the following text : {response}
You need to tell me that how difficult it is to understand.
Answer in one sentence only
''')


fact_check_chain = analysis_prompt|llm | StrOutputParser()

In [22]:
composed_chain = {"response": teacherChain} | fact_check_chain

output = composed_chain.invoke({'school':'phd', 'topics' :'solar energy', 'points': 5 })

print(output)

The text is moderately easy to understand, as it presents clear concepts about solar energy but may require some prior knowledge of technical terms like "photovoltaic cells" and "photoelectric effect."


chain1 =prompt |llm chain2 =response|llm chain3 = prompt | llm
chain4 = response | llm

seq1 = chain1 |chain2
seq2= chain3|chain4

RunnableParallel(c1= seq1, c2=seq2)

In [23]:
#Templates

briefingTemplate = PromptTemplate(
    input_variables=["question"],
    template="""
    You are my special AI Assistant. Act as a Doctor. Answer all queries in 2 sentences.

    Question: {question}
    """
)

detailingTemplate = PromptTemplate(
    input_variables=["question"],
    template="""
    You are my special AI Assistant. Act as a Doctor. Answer all queries as detailed as possible.

    Question: {question}
    """
)

summaryTemplate = PromptTemplate(
    input_variables=['question'],
    template="""
    Give one-line summary for the answer to 

    {question}

    """
)

#Chains

#Seq Chain1

chainA = briefingTemplate | llm
chainB = detailingTemplate | llm

seq1 = chainA | chainB


#Seq Chain2

chainC = summaryTemplate | llm

#Parallel Chain

pChain = RunnableParallel(detailedExplanation = seq1,
                          summary = chainC)


response = pChain.invoke({"question": "What is DNA?"})
response

{'detailedExplanation': AIMessage(content='DNA, or deoxyribonucleic acid, is indeed the fundamental hereditary material in all living organisms, playing a crucial role in the storage and transmission of genetic information. Here’s a more detailed breakdown of its structure and function:\n\n### Structure of DNA\n\n1. **Double Helix**: DNA is composed of two long strands that twist around each other to form a double helix. This structure was famously described by James Watson and Francis Crick in 1953.\n\n2. **Nucleotides**: Each strand of DNA is made up of smaller units called nucleotides. A nucleotide consists of three components:\n   - **A phosphate group**\n   - **A sugar molecule** (deoxyribose in the case of DNA)\n   - **A nitrogenous base** (there are four types: adenine (A), thymine (T), cytosine (C), and guanine (G))\n\n3. **Base Pairing**: The two strands of DNA are held together by hydrogen bonds between the nitrogenous bases. The bases pair specifically: adenine pairs with th